# KNN Model

Train and evaluate a k-Nearest Neighbors classifier.

In [3]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from google.colab import files
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
uploaded = files.upload()

Saving combined_hog_color_lbp.npz to combined_hog_color_lbp (1).npz
Saving label_mapping.pkl to label_mapping (1).pkl


In [8]:
from pathlib import Path
import joblib
import numpy as np

print("Current working folder:", Path.cwd())

# Search everywhere inside /content, including repository folders
npz_files = list(
    Path("/content").rglob("combined_hog_color_lbp*.npz")
)

mapping_files = list(
    Path("/content").rglob("label_mapping*.pkl")
)

print("\nFeature files found:")
for path in npz_files:
    print("-", path)

print("\nMapping files found:")
for path in mapping_files:
    print("-", path)

if not npz_files:
    raise FileNotFoundError(
        "combined_hog_color_lbp.npz was not found."
    )

if not mapping_files:
    raise FileNotFoundError(
        "label_mapping.pkl was not found."
    )

# Select the newest copies
combined_file = max(
    npz_files,
    key=lambda path: path.stat().st_mtime
)

mapping_file = max(
    mapping_files,
    key=lambda path: path.stat().st_mtime
)

feature_data = np.load(
    combined_file,
    allow_pickle=True
)

label_mapping = joblib.load(mapping_file)

print("\nUsing feature file:", combined_file)
print("Using mapping file:", mapping_file)
print("NPZ keys:", feature_data.files)
print("Number of class mappings:", len(label_mapping))

Current working folder: /content/Bird-Species-Classification-ML

Feature files found:
- /content/Bird-Species-Classification-ML/combined_hog_color_lbp.npz
- /content/Bird-Species-Classification-ML/combined_hog_color_lbp (1).npz

Mapping files found:
- /content/Bird-Species-Classification-ML/label_mapping (1).pkl
- /content/Bird-Species-Classification-ML/label_mapping.pkl

Using feature file: /content/Bird-Species-Classification-ML/combined_hog_color_lbp (1).npz
Using mapping file: /content/Bird-Species-Classification-ML/label_mapping (1).pkl
NPZ keys: ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']
Number of class mappings: 20


In [9]:
X_train = feature_data["X_train"]
y_train = feature_data["y_train"]

X_val = feature_data["X_val"]
y_val = feature_data["y_val"]

X_test = feature_data["X_test"]
y_test = feature_data["y_test"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (780, 8206) (780,)
Validation: (167, 8206) (167,)
Testing: (168, 8206) (168,)


## Train a Baseline k-NN Classifier

The k-NN model is first trained using `k = 5`. Feature scaling is applied because k-NN classifies samples using distances between feature vectors.

In [10]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

print("k-NN libraries imported successfully.")


k-NN libraries imported successfully.


In [11]:
# Fit the scaler using only the training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling to validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

print("\nFeature scaling completed successfully.")

Scaled training shape: (780, 8206)
Scaled validation shape: (167, 8206)
Scaled testing shape: (168, 8206)

Feature scaling completed successfully.


In [12]:
baseline_k = 5

baseline_knn = KNeighborsClassifier(
    n_neighbors=baseline_k,
    weights="uniform",
    metric="euclidean",
    n_jobs=-1
)

baseline_knn.fit(
    X_train_scaled,
    y_train
)

print("Baseline k-NN model trained successfully.")
print("Number of neighbors, k:", baseline_k)
print("Training samples:", len(y_train))

Baseline k-NN model trained successfully.
Number of neighbors, k: 5
Training samples: 780


In [13]:
baseline_val_predictions = baseline_knn.predict(
    X_val_scaled
)

baseline_val_accuracy = accuracy_score(
    y_val,
    baseline_val_predictions
)

correct_predictions = (
    baseline_val_predictions == y_val
).sum()

print("Validation images:", len(y_val))
print("Correct predictions:", correct_predictions)

print(
    f"Baseline validation accuracy with k={baseline_k}: "
    f"{baseline_val_accuracy:.4f}"
)

print(
    f"Baseline validation percentage: "
    f"{baseline_val_accuracy * 100:.2f}%"
)

Validation images: 167
Correct predictions: 17
Baseline validation accuracy with k=5: 0.1018
Baseline validation percentage: 10.18%
